# Recurrent Neural Network Language Models

In the previous session we built n-gram language models, counting co-occurrences and estimating probabilities with MLE and smoothing. Those models work reasonably well but have two fundamental limitations:

- They only look back a fixed window of tokens (the order *n*).
- They have no generalization: the model knows nothing about words it hasn't seen in training.

Recurrent Neural Networks (RNNs) address both: they maintain a hidden state that can, in principle, summarize the entire history of tokens seen so far, and they learn distributed representations (embeddings) that allow some generalization across similar words.

In this notebook we will prepare a character-level vocabulary from Austen's *Emma* using Pytorch.

The reason why the model will be character-level is because itt keeps the vocabulary tiny (~60 symbols), avoids OOV problems entirely, and makes the learning dynamics very visible: you can watch the model go from random noise to recognisable English words and sentence structure over training.

## Setup

In [ ]:
!pip install matplotlib
!pip install nltk
!pip install torch

In [ ]:
import nltk
nltk.download('gutenberg', quiet=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import math
import random
import time

from nltk.corpus import gutenberg

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Corpus and Vocabulary

We use Austen's *Emma*, the same corpus from the n-gram session. This time we work at the character level: each character is one token.

In [ ]:
# Load the raw text
raw = gutenberg.raw('austen-emma.txt')

# Light cleanup: collapse whitespace, keep only printable ASCII
import re
text = re.sub(r'\s+', ' ', raw).strip()
text = ''.join(c for c in text if ord(c) < 128)   # printable ASCII only

print(f'Total characters : {len(text):,}')
print(f'Sample           : {repr(text[80:200])}')

In [ ]:
# Build character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

print(f'Vocabulary size  : {vocab_size}')
print(f'Characters       : {repr("".join(chars))}')

In [ ]:
# Encode the full text as a tensor of integer indices
data = torch.tensor([char2idx[c] for c in text], dtype=torch.long)

# Train / validation split (90 / 10)
training_frac = 0.9
split = int(training_frac * len(data))
train_data = data[:split]
val_data   = data[split:]

# Reduce training data (to reduce computational requirements)
train_data = train_data[:100000]

print(f'Train tokens : {len(train_data):,}')
print(f'Val tokens   : {len(val_data):,}')

## Dataset

We train the model to predict the next character given the previous `SEQ_LEN` characters. We slide a window of length `SEQ_LEN + 1` over the text: the first `SEQ_LEN` characters are the input, `x`, and the last `SEQ_LEN` characters (shifted by one) are the target, `y`.

```
text:   E m m a   w a s   c l e v e r
x:      E m m a   w a s   c l e v e
y:      m m a   w a s   c l e v e r
```

This is the standard language modelling objective: predict the next token at every position.

In [ ]:
SEQ_LEN    = 100   # number of characters per training sequence
BATCH_SIZE = 64

class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx          : idx + self.seq_len]
        y = self.data[idx + 1      : idx + self.seq_len + 1]
        return x, y

train_dataset = CharDataset(train_data, SEQ_LEN)
val_dataset   = CharDataset(val_data,   SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Training batches   : {len(train_loader)}')
print(f'Validation batches : {len(val_loader)}')

# Inspect one batch
xb, yb = next(iter(train_loader))
print(f'\nBatch x shape : {xb.shape}  (batch_size × seq_len)')
print(f'Batch y shape : {yb.shape}')
print(f'\nFirst sequence (x): {repr("".join(idx2char[i.item()] for i in xb[0][:40]))}')
print(f'First target   (y): {repr("".join(idx2char[i.item()] for i in yb[0][:40]))}')

## The LSTM Language Model

Our model has three components:

| Component | Purpose |
|---|---|
| **Embedding** | Maps each character index to a dense vector |
| **LSTM** | Processes the sequence and updates a hidden state |
| **Linear head** | Projects the hidden state to logits over the vocabulary |

At each time step $t$, the LSTM takes the embedding of $x_t$ and the previous hidden state $(h_{t-1}, c_{t-1})$, and produces a new hidden state $(h_t, c_t)$. The linear head converts $h_t$ to a probability distribution over the next character.

Compare this to the n-gram model: instead of a lookup table of counts, we have a differentiable function with millions of parameters that is learned from data.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.embed      = nn.Embedding(vocab_size, embed_dim)
        self.lstm       = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.drop       = nn.Dropout(dropout)
        self.head       = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        emb = self.drop(self.embed(x))          # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)    # out: (batch, seq_len, hidden_dim)
        out = self.drop(out)
        logits = self.head(out)                 # (batch, seq_len, vocab_size)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
        return (h, c)

# Hyperparameters
EMBED_DIM  = 64
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT    = 0.3

model = CharLSTM(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters : {total_params:,}')
print(model)

## Training

We optimize **cross-entropy loss** at every character position (the same objective as minimizing perplexity). We detach the hidden state between batches (`hidden.detach()`) to avoid backpropagating through the entire corpus history, which would be computationally infeasible. This is called truncated backpropagation through time (TBPTT).

In [ ]:
def detach_hidden(hidden):
    """Detach hidden state from computation graph (TBPTT)."""
    return tuple(h.detach() for h in hidden)


def run_epoch(model, loader, optimizer=None, clip=1.0):
    """Run one full epoch. If optimiser is None, run in eval mode."""
    training = optimizer is not None
    model.train(training)

    total_loss   = 0.0
    total_tokens = 0

    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            hidden  = model.init_hidden(xb.size(0), DEVICE)

            logits, hidden = model(xb, hidden)
            hidden = detach_hidden(hidden)

            # Reshape for cross-entropy: (batch*seq_len, vocab) vs (batch*seq_len,)
            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                yb.reshape(-1)
            )

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), clip)  # gradient clipping
                optimizer.step()

            total_loss   += loss.item() * yb.numel()
            total_tokens += yb.numel()

    avg_loss = total_loss / total_tokens
    return avg_loss, math.exp(avg_loss)   # (loss, perplexity)

In [ ]:
# Training configuration
NUM_EPOCHS = 3
LR         = 3e-3

optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

history = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []}

print(f'{"Epoch":>6}  {"Train Loss":>11}  {"Train PPL":>10}  {"Val Loss":>10}  {"Val PPL":>9}  {"Time":>6}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_ppl = run_epoch(model, train_loader, optimizer=optimizer)
    va_loss, va_ppl = run_epoch(model, val_loader,   optimizer=None)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_ppl'].append(tr_ppl)
    history['val_ppl'].append(va_ppl)

    elapsed = time.time() - t0
    print(f'{epoch:>6}  {tr_loss:>11.4f}  {tr_ppl:>10.2f}  {va_loss:>10.4f}  {va_ppl:>9.2f}  {elapsed:>5.1f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs, history['val_loss'],   label='Val',   marker='s')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_ppl'], label='Train', marker='o')
axes[1].plot(epochs, history['val_ppl'],   label='Val',   marker='s')
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PPL')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFinal validation perplexity: {history["val_ppl"][-1]:.2f}')

## Text Generation with Temperature Sampling

After training, we can generate text autoregressively: we feed a prompt, get a probability distribution over the next character, sample from it, and append the sampled character to the context.

The temperature parameter $T$ controls how sharp or flat the distribution is:

$$P(c) \propto \exp\left(\frac{\text{logit}(c)}{T}\right)$$

- $T \to 0$: greedy
- $T = 1$: standard sampling
- $T > 1$: flatter distribution

In [ ]:
@torch.no_grad()
def generate(model, prompt, num_chars=300, temperature=1.0):
    model.eval()

    # Encode prompt
    context = [char2idx.get(c, 0) for c in prompt]
    x       = torch.tensor(context, dtype=torch.long, device=DEVICE).unsqueeze(0)  # (1, len)

    # Warm up the hidden state with the prompt
    hidden = model.init_hidden(1, DEVICE)
    logits, hidden = model(x, hidden)

    generated = prompt

    # Use the last logit as starting point
    last_logit = logits[0, -1, :]   # (vocab_size,)

    for _ in range(num_chars):
        # Apply temperature and sample
        probs   = F.softmax(last_logit / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        next_ch = idx2char[next_id]
        generated += next_ch

        # Feed the new character back
        x = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)
        logits, hidden = model(x, hidden)
        last_logit = logits[0, 0, :]

    return generated

In [ ]:
PROMPT = "Emma Woodhouse, handsome, clever, and rich, with a comfortable home"

for temp in [0.5, 1.0, 1.5]:
    print(f'\n{"─" * 60}')
    print(f'  Temperature = {temp}')
    print(f'{"─" * 60}')
    print(generate(model, PROMPT, num_chars=250, temperature=temp))

In [ ]:
# Try your own prompt!
my_prompt = "Mr. Knightley looked at her with"
print(generate(model, my_prompt, num_chars=300, temperature=0.8))

## Perplexity Comparison: LSTM vs. N-gram

To make a fair comparison with n-gram models, we need to compute perplexity at the word level (or at least compare the same metric). Our character-level LSTM perplexity is over characters; the n-gram perplexity from last session was over words.

Instead of converting units, let's compare them qualitatively and focus on what the numbers mean:

| Model | Perplexity (chars / words) | Notes |
|---|---|---|
| Trigram MLE | ∞ (unseen n-grams) | Blows up on OOV |
| Trigram KN | ~60–90 (word-level) | Good smoothing, fixed context |
| LSTM (this notebook) | ~?? (char-level) | Longer context, generalises |

A character-level perplexity of around 3–5 is typical for a well-trained small LSTM on a literary corpus (there are approximately 60 characters, and the model is much better than random). Word-level perplexity for a comparable LSTM on Emma would typically be in the range of 50–120, similar to or better than n-gram models, but the real advantage is in longer coherence and no OOV problem.

In [ ]:
# Compute final validation perplexity
val_loss, val_ppl = run_epoch(model, val_loader, optimizer=None)

print(f'Character-level validation perplexity : {val_ppl:.2f}')
print(f'  (out of {vocab_size} possible characters)')
print(f'  Bits per character                  : {val_loss / math.log(2):.3f}')

# Naive baseline: uniform distribution over vocabulary
baseline_ppl = vocab_size
print(f'\nBaseline perplexity (uniform)         : {baseline_ppl}')
print(f'Improvement factor                    : {baseline_ppl / val_ppl:.1f}x')

## Limitations of RNNs

Our LSTM produces coherent local text, but it has well-known structural limitations:

### The vanishing gradient problem

During the training procedure (using backpropagation through time) the gradient of the loss with respect to early timesteps is computed by repeatedly multiplying the same recurrent weight matrix at each step. So to reach timestep 0 in a sequence of length T, the gradient passes through T matrix multiplications. If the weights are slightly smaller than 1 in magnitude, this repeated multiplication causes the gradient to shrink exponentially: after 50 steps, a factor of 0.9 per step becomes 0.9⁵⁰ ≈ 0.005, and the weights responsible for processing early tokens receive an update so tiny they effectively never change. The model therefore fails to learn any dependency that spans more than a handful of timesteps, since the error signal from a prediction at the end of the sequence simply never reaches the parameters that processed the beginning.

### Sequential computation

RNNs process tokens one at a time, left to right. This means that the training procedure cannot be parallelized along the sequence dimension, which on modern GPUs (designed for massive parallelism), is a significant bottleneck.